<a href="https://colab.research.google.com/github/data4class/Teaching/blob/main/Isolation_Forest_model_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Isolation forest model

Isolation Forest is an unsupervised anomaly detection algorithm that works on a simple principle: anomalies are easier to isolate than normal points.

**How it works:**

Random partitioning - Randomly selects features and split values to create binary trees

Isolation process - Keeps splitting data points until each point is isolated in its own leaf

Path length - Measures how many splits it took to isolate each point

Anomaly scoring - Points that require fewer splits (shorter paths) are more likely to be anomalies

**Key insight:**

Normal points are densely clustered, so they need many splits to separate from their neighbors. Anomalies are sparse/isolated, so they get separated quickly with fewer splits.

**Advantages:**

Fast and scalable

Works well in high dimensions

No need to define "normal" behavior

Handles large datasets efficiently

**Use cases:** Fraud detection, network intrusion detection, quality control, outlier detection in any dataset.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
import matplotlib.pyplot as plt

# Load the data
## Load and preprocess data (same as before)
# Step 1: Load and preprocess the data
file_id = '1hctEm-25IIt9CPUduQExa1COSyS9o7_G'
url = f'https://drive.google.com/uc?id={file_id}&export=download'

# Load with pipe separator and no header
data = pd.read_csv(url, sep='|', header=None)

# Assign proper column names (0-indexed)
# Column 4 (index 3): price, Column 5 (index 4): volume, Column 6 (index 5): timestamp
data = data.iloc[:, [5, 3, 4]]
data.columns = ['timestamp', 'price', 'quantity']


# Convert timestamp to datetime, letting pandas infer the format
data['timestamp'] = pd.to_datetime(data['timestamp'], format='mixed')

# Normalize the data
numerical_data = data[['price', 'quantity']]
scaler = StandardScaler()
normalized_numerical_data = scaler.fit_transform(numerical_data)

# Create and fit the Isolation Forest model
contamination = 0.0005  # This assumes 0.1% of the data points are anomalies
iso_forest = IsolationForest(contamination=contamination)
iso_forest.fit(normalized_numerical_data)

# Predict anomalies
anomaly_labels = iso_forest.predict(normalized_numerical_data)
data['anomaly'] = anomaly_labels

# Separate anomalies and normal points
anomalies = data[data['anomaly'] == -1]
normal = data[data['anomaly'] == 1]

# Print summary of anomalies
print(f"Number of anomalies detected: {len(anomalies)}")
print("\nSample of detected anomalies:")
print(anomalies)

# Visualize the results
plt.figure(figsize=(12, 6))
plt.scatter(normal['timestamp'], normal['price'], color='blue', label='Normal', alpha=0.5)
plt.scatter(anomalies['timestamp'], anomalies['price'], color='red', label='Anomaly', alpha=0.5)
plt.xlabel('Timestamp')
plt.ylabel('Price')
plt.title('Anomaly Detection using Isolation Forest')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Additional visualization: Price vs Quantity
plt.figure(figsize=(12, 6))
plt.scatter(normal['price'], normal['quantity'], color='blue', label='Normal', alpha=0.5)
plt.scatter(anomalies['price'], anomalies['quantity'], color='red', label='Anomaly', alpha=0.5)
plt.xlabel('Price')
plt.ylabel('Quantity')
plt.title('Anomaly Detection: Price vs Quantity')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
print(data.head())

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Load the data
# If the data is not available locally, you'll need to download it first
url = 'https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv'
df = pd.read_csv(url, nrows = 10000)

# Separate features and target
X = df.drop('Class', axis=1)  # Assuming 'Class' is the target column
y = df['Class']

# Normalize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Create and fit the Isolation Forest model
iso_forest = IsolationForest(contamination=0.01, random_state=42)
iso_forest.fit(X_scaled)

# Get anomaly scores
anomaly_scores = iso_forest.decision_function(X_scaled)

# Add anomaly scores to the dataframe
df['Anomaly_Score'] = anomaly_scores

# Sort by anomaly score (ascending) to get the most anomalous transactions
df_sorted = df.sort_values('Anomaly_Score')

# Calculate the number of rows to print (1% of the data)
num_anomalies = int(len(df) * 0.003)

# Print the top 0.1% of anomalies
print(f"Top {num_anomalies} anomalies:")
print(df_sorted.head(num_anomalies))

In [ ]:
print(sum(df_sorted.head(num_anomalies)['Class'] == 1))